# Persistent Foundry Agent

Creates a persistent agent on Azure AI Foundry, sends a user message, and streams the response back in real time.

In [ ]:
%pip install azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import AgentStreamEvent
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  # loads .env from repo root

endpoint = os.environ["AZURE_FOUNDRY_ENDPOINT"]

client = AIProjectClient(endpoint=endpoint, credential=DefaultAzureCredential())
print("Client ready.")

In [ ]:
# Create a persistent Foundry agent
agent = client.agents.create_agent(
    model="gpt-4.1",
    name="WriterAgent",
    instructions="You are a helpful agent that writes engaging book stories.",
)
print(f"Agent created: {agent.id}")

In [ ]:
# Create a conversation thread and send a user message
thread = client.agents.create_thread()
print(f"Thread created: {thread.id}")

client.agents.create_message(
    thread_id=thread.id,
    role="user",
    content="Write a short story about a robot who discovers music.",
)

In [ ]:
# Stream the agent's response
print("--- Agent Response ---\n")

with client.agents.create_stream(thread_id=thread.id, agent_id=agent.id) as stream:
    for event_type, event_data, _ in stream:
        if event_type == AgentStreamEvent.THREAD_MESSAGE_DELTA:
            for block in event_data.delta.content:
                if block.type == "text":
                    print(block.text.value, end="", flush=True)

In [ ]:
# Cleanup: delete the agent when done
client.agents.delete_agent(agent_id=agent.id)
print("\nAgent deleted.")